In [ ]:
import pandas as pd
import numpy as np
import json
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Machine Learning imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
import joblib


print("\nLoading audio features dataset...")
audio_df = pd.read_csv('audio_features_clean.csv')
print(f"Loaded {len(audio_df):,} tracks with emotions")

print(f"\nCurrent emotion distribution:")
emotion_dist = audio_df['emotion'].value_counts()
for emotion, count in emotion_dist.items():
    print(f"   {emotion:12}: {count:6,} ({count/len(audio_df)*100:.1f}%)")



def enhanced_emotion_mapping(row):
    """Better emotion mapping using multiple audio features"""
    valence = row.get('valence', 0.5)
    energy = row.get('energy', 0.5)
    danceability = row.get('danceability', 0.5)
    acousticness = row.get('acousticness', 0.5)
    tempo = row.get('tempo', 120)
    instrumentalness = row.get('instrumentalness', 0)
    
    # Calculate combined scores
    happy_score = (valence * 0.4 + energy * 0.3 + danceability * 0.3)
    sad_score = ((1 - valence) * 0.5 + (1 - energy) * 0.3 + acousticness * 0.2)
    energetic_score = (energy * 0.5 + danceability * 0.3 + (tempo > 120) * 0.2)
    chill_score = ((1 - energy) * 0.4 + acousticness * 0.3 + (tempo < 100) * 0.3)
    party_score = (danceability * 0.4 + energy * 0.4 + valence * 0.2)
    calm_score = ((1 - energy) * 0.5 + acousticness * 0.3 + instrumentalness * 0.2)
    romantic_score = (valence * 0.4 + acousticness * 0.3 + (1 - energy) * 0.3)
    focus_score = (instrumentalness * 0.4 + (1 - danceability) * 0.3 + (tempo < 100) * 0.3)
    
    scores = {
        'happy': happy_score,
        'sad': sad_score,
        'energetic': energetic_score,
        'chill': chill_score,
        'party': party_score,
        'calm': calm_score,
        'romantic': romantic_score,
        'focus': focus_score
    }
    
    # Get top 2 emotions
    sorted_emotions = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    primary_emotion = sorted_emotions[0][0]
    secondary_emotion = sorted_emotions[1][0]
    confidence = sorted_emotions[0][1] - sorted_emotions[1][1]
    
    return primary_emotion, secondary_emotion, confidence

# Apply enhanced mapping
print("\nApplying enhanced emotion mapping...")
enhanced_emotions = []
for _, row in tqdm(audio_df.iterrows(), total=len(audio_df), desc="Mapping emotions"):
    primary, secondary, confidence = enhanced_emotion_mapping(row)
    enhanced_emotions.append({
        'primary_emotion': primary,
        'secondary_emotion': secondary,
        'emotion_confidence': confidence
    })

audio_df['primary_emotion'] = [e['primary_emotion'] for e in enhanced_emotions]
audio_df['secondary_emotion'] = [e['secondary_emotion'] for e in enhanced_emotions]
audio_df['emotion_confidence'] = [e['emotion_confidence'] for e in enhanced_emotions]

print(f"\nEnhanced emotion distribution:")
print(audio_df['primary_emotion'].value_counts())



feature_cols = ['valence', 'energy', 'danceability', 'acousticness', 
                'instrumentalness', 'tempo', 'loudness', 'speechiness', 
                'liveness', 'key', 'mode']

# Prepare features
X = audio_df[feature_cols].fillna(0)
y = audio_df['primary_emotion']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# Second split: 15% validation, 15% test (from the 30% temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nDataset Split:")
print(f"   Training:   {len(X_train):,} samples ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"   Validation: {len(X_val):,} samples ({len(X_val)/len(X_scaled)*100:.1f}%)")
print(f"   Test:       {len(X_test):,} samples ({len(X_test)/len(X_scaled)*100:.1f}%)")

print(f"\nTraining set emotion distribution:")
train_emotions = le.inverse_transform(y_train)
train_counts = Counter(train_emotions)
for emotion, count in train_counts.most_common():
    print(f"   {emotion:12}: {count:5,} ({count/len(train_emotions)*100:.1f}%)")



# Initialize model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# 5-fold cross-validation on training set
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print(f"   CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Train on full training set
print("\nTraining on full training set...")
rf.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred = rf.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
print(f"\nValidation Results:")
print(f"   Accuracy: {val_accuracy:.4f}")
print(f"   Weighted F1: {val_f1:.4f}")

print(f"\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=le.classes_))



y_test_pred = rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nTest Results:")
print(f"   Accuracy: {test_accuracy:.4f}")
print(f"   Weighted F1: {test_f1:.4f}")

print(f"\nTest Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

# Confusion Matrix
print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
print(feature_importance.to_string(index=False))



# Create lookup dictionaries
exact_lookup = {}
track_name_lookup = {}

def safe_str(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, float):
        return ""
    return str(value).lower().strip()

for _, row in audio_df.iterrows():
    track_name = safe_str(row['track_name'])
    artist_name = safe_str(row['artist_name'])
    
    # Extract primary artist
    primary_artist = artist_name.split(',')[0].strip() if ',' in artist_name else artist_name
    
    # Exact match key
    exact_key = f"{track_name}|{primary_artist}"
    exact_lookup[exact_key] = row.to_dict()
    
    # Track name lookup
    if track_name not in track_name_lookup:
        track_name_lookup[track_name] = []
    track_name_lookup[track_name].append(row.to_dict())

print(f"Created {len(exact_lookup):,} exact matches")
print(f"Created {len(track_name_lookup):,} track name entries")



# Load listening data
try:
    with open('music_listening_data.json', 'r') as f:
        listening_data = json.load(f)
    print(f"Loaded {len(listening_data):,} events")
except:
    print("No listening data found, using sample...")
    # Create sample if none exists
    listening_data = []

if listening_data:
    processed_events = []
    match_stats = {'exact': 0, 'track_name': 0, 'model_prediction': 0, 'no_match': 0}
    predictions = []
    
    for event in tqdm(listening_data, desc="Processing"):
        track_name = safe_str(event.get('title', event.get('track_name', '')))
        artist_name = safe_str(event.get('text', event.get('artist', '')))
        
        # Extract primary artist
        primary_artist = artist_name.split(',')[0].strip() if ',' in artist_name else artist_name
        
        # Try exact match
        exact_key = f"{track_name}|{primary_artist}"
        
        if exact_key in exact_lookup:
            match_stats['exact'] += 1
            track_info = exact_lookup[exact_key]
            event['emotion'] = track_info['primary_emotion']
            event['secondary_emotion'] = track_info['secondary_emotion']
            event['confidence'] = track_info['emotion_confidence']
            event['match_type'] = 'exact'
            
        elif track_name in track_name_lookup:
            match_stats['track_name'] += 1
            track_info = track_name_lookup[track_name][0]
            event['emotion'] = track_info['primary_emotion']
            event['secondary_emotion'] = track_info['secondary_emotion']
            event['confidence'] = track_info['emotion_confidence'] * 0.8
            event['match_type'] = 'track_name'
            
        else:
            # Use model prediction based on nothing? Actually we don't have features
            # For now, mark as unknown
            match_stats['no_match'] += 1
            event['emotion'] = 'unknown'
            event['confidence'] = 0
            event['match_type'] = 'none'
        
        processed_events.append(event)
        if event['emotion'] != 'unknown':
            predictions.append(event['emotion'])
    
    # Results
    print(f"\nMatching Results:")
    total = len(processed_events)
    for match_type, count in match_stats.items():
        print(f"   {match_type:18}: {count:6,} ({count/total*100:.1f}%)")
    
    if predictions:
        print(f"\nEmotion Distribution (from {len(predictions):,} matched tracks):")
        emotion_counts = Counter(predictions)
        for emotion, count in emotion_counts.most_common():
            print(f"   {emotion:12}: {count:6,} ({count/len(predictions)*100:.1f}%)")
    
    # Save processed events
    with open('processed_listening_data_final.json', 'w') as f:
        sample_size = min(10000, len(processed_events))
        json.dump(processed_events[:sample_size], f, indent=2)
    print(f"\nSaved {sample_size:,} sample events to processed_listening_data_final.json")
    
    # Save full dataset as CSV
    df = pd.DataFrame(processed_events)
    df.to_csv('full_processed_listening_data.csv', index=False)
    print(f"Saved all {len(df):,} events to full_processed_listening_data.csv")



# Save model
joblib.dump(rf, 'random_forest_emotion_model.pkl')
print(f"Saved model to random_forest_emotion_model.pkl")

# Save scaler
joblib.dump(scaler, 'feature_scaler.pkl')
print(f"Saved scaler to feature_scaler.pkl")

# Save label encoder
joblib.dump(le, 'label_encoder.pkl')
print(f"Saved label encoder to label_encoder.pkl")

# Save feature columns
joblib.dump(feature_cols, 'feature_columns.pkl')
print(f"Saved feature columns to feature_columns.pkl")

print(f"\nModel Performance:")
print(f"   Training Size:     {len(X_train):,} samples")
print(f"   Validation Size:   {len(X_val):,} samples")
print(f"   Test Size:         {len(X_test):,} samples")
print(f"   Cross-Validation:  {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"   Validation Accuracy: {val_accuracy:.4f}")
print(f"   Test Accuracy:     {test_accuracy:.4f}")
print(f"   Test F1 Score:     {test_f1:.4f}")

print(f"\nBest Performing Emotions (by F1-score):")
class_report = classification_report(y_test, y_test_pred, target_names=le.classes_, output_dict=True)
for emotion in le.classes_:
    f1 = class_report[emotion]['f1-score']
    print(f"   {emotion:12}: {f1:.3f}")

print(f"\nFiles Saved:")
print(f"   • random_forest_emotion_model.pkl - Trained model")
print(f"   • feature_scaler.pkl - Feature scaler")
print(f"   • label_encoder.pkl - Label encoder")
print(f"   • feature_columns.pkl - Feature names")
print(f"   • processed_listening_data_final.json - Sample processed events")
print(f"   • full_processed_listening_data.csv - All processed events")


Loading audio features dataset...
Loaded 95,235 tracks with emotions

Current emotion distribution:
   chill       : 13,112 (13.8%)
   calm        : 12,509 (13.1%)
   romantic    : 12,306 (12.9%)
   happy       : 12,196 (12.8%)
   energetic   : 11,909 (12.5%)
   sad         : 11,740 (12.3%)
   focus       : 11,313 (11.9%)
   party       : 10,150 (10.7%)


Applying enhanced emotion mapping...


Mapping emotions: 100%|██████████| 95235/95235 [00:23<00:00, 4057.85it/s]



Enhanced emotion distribution:
primary_emotion
energetic    31217
sad          16272
chill        14081
party        13493
happy         8088
calm          6017
focus         4318
romantic      1749
Name: count, dtype: int64


Dataset Split:
   Training:   66,664 samples (70.0%)
   Validation: 14,285 samples (15.0%)
   Test:       14,286 samples (15.0%)

Training set emotion distribution:
   energetic   : 21,852 (32.8%)
   sad         : 11,390 (17.1%)
   chill       : 9,857 (14.8%)
   party       : 9,445 (14.2%)
   happy       : 5,661 (8.5%)
   calm        : 4,212 (6.3%)
   focus       : 3,023 (4.5%)
   romantic    : 1,224 (1.8%)


Performing 5-fold cross-validation...
   CV Accuracy: 0.9298 (+/- 0.0039)

Training on full training set...

Validation Results:
   Accuracy: 0.9329
   Weighted F1: 0.9341

Validation Classification Report:
              precision    recall  f1-score   support

        calm       0.89      0.97      0.93       902
       chill       0.96      0.96      0.96

Processing: 100%|██████████| 93653/93653 [00:00<00:00, 128630.81it/s]



Matching Results:
   exact             : 93,653 (100.0%)
   track_name        :      0 (0.0%)
   model_prediction  :      0 (0.0%)
   no_match          :      0 (0.0%)

Emotion Distribution (from 93,653 matched tracks):
   energetic   : 32,146 (34.3%)
   sad         : 15,747 (16.8%)
   party       : 13,594 (14.5%)
   chill       : 13,035 (13.9%)
   happy       :  8,327 (8.9%)
   calm        :  5,211 (5.6%)
   focus       :  3,892 (4.2%)
   romantic    :  1,701 (1.8%)

Saved 10,000 sample events to processed_listening_data_final.json
Saved all 93,653 events to full_processed_listening_data.csv

Saved model to random_forest_emotion_model.pkl
Saved scaler to feature_scaler.pkl
Saved label encoder to label_encoder.pkl
Saved feature columns to feature_columns.pkl


Model Performance:
   Training Size:     66,664 samples
   Validation Size:   14,285 samples
   Test Size:         14,286 samples
   Cross-Validation:  0.9298 (+/- 0.0039)
   Validation Accuracy: 0.9329
   Test Accuracy:     0.9